In [1]:
import json
import os 

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "tennie2010two")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "tennie2010two_not_original_format.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)

In [3]:
data_list = df.values.tolist()
column_name = df.columns.values.tolist()
output = []
for line in data_list: 
    for value,name in zip(line[4:],column_name[4:]): 
        output.append([line[0], line[1],line[2],line[3], value, name]) 
longdf = pd.DataFrame(output, columns=['participant', 'rearing','sex', 'age', 'values', 'temp_columns'])

In [4]:
longdf['temp_columns'].replace('set ', '', inplace=True, regex=True)
longdf['temp_columns'].replace(' condition', '', inplace=True, regex=True)
longdf['temp_columns'].unique
longdf[['set','condition', 'temp_header']] = longdf['temp_columns'].str.split('_',expand=True, n=2)
longdf.columns

Index(['participant', 'rearing', 'sex', 'age', 'values', 'temp_columns', 'set',
       'condition', 'temp_header'],
      dtype='object')

In [5]:
no_bl = longdf[~longdf.condition.str.contains("bl")] 
bl = longdf[longdf.condition.str.contains("bl")]

matching = no_bl[no_bl.temp_header.str.contains("matching_score")]
success = no_bl[no_bl.temp_header.str.contains("first_trial_success")]
m_list =matching[['participant', 'rearing', 'sex', 'age', 'values',  'set',
       'condition']].values.tolist() ##you only need one set of repeating values
s_list = success[[ 'values']].values.tolist() ##this is what is new
# print(len(m_list)) 
# print(len(s_list)) 
combined_lol = [lol_1+lol_2 for lol_1,lol_2 in zip(m_list,s_list)]

zipdf = pd.DataFrame(combined_lol, columns=['participant', 'rearing', 'sex', 'age', 'matching_scores',  'set',
       'condition', 'first_success'])
# combined_lol = combined_lol

In [6]:
data_frames = [zipdf, bl]
for index, x in enumerate(data_frames):
    x.columns = map(str.lower, x.columns)
    x=x.applymap(lambda s: s.lower() if type(s) == str else s) 
    x.rename(columns={"values": "first_success", 
                      'sex':'sex_original'}, inplace=True)
    x['study_id']="tennie2010two"
    data_frames[index]=x
fulldf = pd.concat(data_frames, ignore_index=True, sort=False)

In [7]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
fulldf['participant'] = fulldf['participant'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    fulldf['participant'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)    
fulldf= fulldf.merge(apedf,left_on='participant', right_on='name', how='left')

In [8]:
fulldf=fulldf.sort_values(by = ['participant','set'])
# fulldf.columns

In [9]:
fulldf['first_success'].replace('-', '', inplace=True, regex=True)
fulldf['rearing'].replace('\*', '', inplace=True, regex=True)
fulldf['matching_scores'].replace('[-]*$', '', inplace=True, regex=True)

condition_list = [['fm','full_model'],
                  ['in','intention'],
                  ['es','end_state'],
                  ['bl','baseline']]
for x,y in condition_list:
        fulldf['condition'].replace(x, y, inplace=True, regex=True)

In [10]:
fulldf.rename(columns={"age": "age_in_years"}, inplace=True)

In [11]:


studyID_standardized=fulldf[['study_id', 'participant', 'age_in_years','sex', 
        'species',  'set','condition', 'first_success','matching_scores']]
comp_out_path_stand = os.path.join(out_pathway, 'tennie2010two_standardized.csv')
studyID_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)


names =studyID_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
studyID_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'tennie2010two_glossary.csv')
studyID_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)
